# NosoGraph MVP Demo — CSV → Neo4j (Virowatch Schema)

Loads all data from `example/csv/` into a Neo4j graph using the **Specimen-centric virowatch schema**.
No `nosograph-py` library imports — just the `neo4j` Python driver and `pandas`.

**Run this notebook from the `example/` directory:**
```bash
cd NosoGraph/example && jupyter notebook nosograph_demo.ipynb
```

| # | Section | What it does |
|---|---|---|
| 0 | Connect | Open the Neo4j driver |
| 1 | Constraints | Apply schema uniqueness constraints |
| 2 | Reference data | Departments, Wards, Organisms, Reference Genomes |
| 3 | Clinical entities | Patients, OPD Visits, Admissions, Specimens |
| 4 | Lab data | Lab Results, HIV Viral Loads |
| 5 | Relationships | Wire all nodes together |
| 6 | Genomic provenance | SequencingRun → Assembly → Contig + VariantCallingRun → Variants |
| 7 | Explore | Node counts, patient chain, variant traversal |

**Schema:** matches the live `virowatch` database — Specimen is the anchor for all genomic data.
The traversal spine is: `Specimen -[:HAS_ASSEMBLY]-> Assembly -[:HAS_VARIANT_CALLING_RUN]-> VariantCallingRun -[:FOUND]-> Variant`

**Prerequisites:** `pip install neo4j pandas python-dotenv`

---
## 0. Connect

In [ ]:
from pathlib import Path
import os
import pandas as pd
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
_auth = os.getenv("NEO4J_AUTH", "neo4j/password")
_user, _pwd = _auth.split("/", 1)

driver = GraphDatabase.driver(NEO4J_URI, auth=(_user, _pwd))
driver.verify_connectivity()
print(f"Connected to {NEO4J_URI}")

CSV_DIR = Path("csv")


def run(cypher, **params):
    with driver.session() as s:
        return list(s.run(cypher, params))


def maybe_int(v):
    try:
        return int(float(v)) if str(v).strip() not in ("", "nan") else None
    except (ValueError, TypeError):
        return None


def maybe_float(v):
    try:
        return float(v) if str(v).strip() not in ("", "nan") else None
    except (ValueError, TypeError):
        return None


def clean(v):
    s = str(v).strip()
    return None if s in ("", "nan") else s


print("Ready.")

---
## 1. Schema Constraints

Idempotent — safe to re-run. Property names match the virowatch database exactly.

In [ ]:
CONSTRAINTS = [
    # Reference / lookup nodes
    "CREATE CONSTRAINT dept_id_unique    IF NOT EXISTS FOR (d:Department)      REQUIRE d.department_id IS UNIQUE",
    "CREATE CONSTRAINT ward_id_unique    IF NOT EXISTS FOR (w:Ward)             REQUIRE w.ward_id IS UNIQUE",
    "CREATE CONSTRAINT organism_taxid    IF NOT EXISTS FOR (o:Organism)         REQUIRE o.taxid IS UNIQUE",
    "CREATE CONSTRAINT refg_acc_unique   IF NOT EXISTS FOR (r:ReferenceGenome)  REQUIRE r.accession IS UNIQUE",
    # Clinical
    "CREATE CONSTRAINT patient_id_unique IF NOT EXISTS FOR (p:Patient)          REQUIRE p.patient_id IS UNIQUE",
    "CREATE CONSTRAINT opd_id_unique     IF NOT EXISTS FOR (v:OpdVisit)         REQUIRE v.visit_id IS UNIQUE",
    "CREATE CONSTRAINT adm_id_unique     IF NOT EXISTS FOR (a:Admission)        REQUIRE a.admission_id IS UNIQUE",
    "CREATE CONSTRAINT specimen_id_unique IF NOT EXISTS FOR (s:Specimen)        REQUIRE s.specimen_id IS UNIQUE",
    # Lab
    "CREATE CONSTRAINT lab_result_id     IF NOT EXISTS FOR (lr:LabResult)       REQUIRE lr.result_id IS UNIQUE",
    "CREATE CONSTRAINT hiv_vl_id         IF NOT EXISTS FOR (vl:HIVViralLoad)    REQUIRE vl.result_id IS UNIQUE",
    # Genomic
    "CREATE CONSTRAINT seq_run_id_unique IF NOT EXISTS FOR (sr:SequencingRun)   REQUIRE sr.run_id IS UNIQUE",
    "CREATE CONSTRAINT assembly_id_unique IF NOT EXISTS FOR (a:Assembly)        REQUIRE a.assembly_id IS UNIQUE",
    "CREATE CONSTRAINT contig_id_unique  IF NOT EXISTS FOR (c:Contig)           REQUIRE c.contig_id IS UNIQUE",
    "CREATE CONSTRAINT vcr_run_id_unique IF NOT EXISTS FOR (v:VariantCallingRun) REQUIRE v.run_id IS UNIQUE",
    "CREATE CONSTRAINT feature_locus_tag IF NOT EXISTS FOR (f:Feature)          REQUIRE f.locus_tag IS UNIQUE",
]

for cql in CONSTRAINTS:
    run(cql)

print("Constraints applied.")

---
## 2. Reference Data

`Departments.csv`, `Wards.csv`, `Organisms.csv`, `ReferenceGenomes.csv`

**Virowatch property names:**
- `Organism.name` (not `sciname`)
- `ReferenceGenome.accession` (not `accession_no`)

In [ ]:
# --- Departments ---
df = pd.read_csv(CSV_DIR / "Departments.csv").fillna("")
for _, r in df.iterrows():
    run("""
        MERGE (d:Department {department_id: $department_id})
        ON CREATE SET d.name = $name, d.description = $description, d.created_at = datetime()
    """, department_id=r.department_id, name=r["name"], description=r.description)
print(f"Departments loaded: {len(df)}")

# --- Wards ---
df = pd.read_csv(CSV_DIR / "Wards.csv").fillna("")
for _, r in df.iterrows():
    run("""
        MERGE (w:Ward {ward_id: $ward_id})
        ON CREATE SET w.name = $name, w.ward_type = $ward_type, w.description = $description, w.created_at = datetime()
    """, ward_id=r.ward_id, name=r["name"], ward_type=r.ward_type, description=r.description)
print(f"Wards loaded: {len(df)}")

# --- Organisms (virowatch: name, taxid, strain) ---
df = pd.read_csv(CSV_DIR / "Organisms.csv").fillna("")
for _, r in df.iterrows():
    run("""
        MERGE (o:Organism {taxid: $taxid})
        ON CREATE SET o.name = $name
    """, taxid=str(r.taxid), name=r.sciname)
print(f"Organisms loaded: {len(df)}")

# --- Reference Genomes (virowatch: accession, not accession_no) ---
df = pd.read_csv(CSV_DIR / "ReferenceGenomes.csv").fillna("")
for _, r in df.iterrows():
    run("""
        MERGE (rg:ReferenceGenome {accession: $accession})
        ON CREATE SET
            rg.name           = $name,
            rg.molecular_type = $molecular_type,
            rg.strain         = $strain
    """, accession=r.accession_no, name=r["name"],
         molecular_type=r.molecular_type, strain=r.strain)
print(f"Reference genomes loaded: {len(df)}")

---
## 3. Clinical Entities

`Patients.csv`, `OpdVisits.csv`, `Admissions.csv`, `Specimens.csv`

**Virowatch:** `Specimen` is the primary anchor for genomic data (no `Sample` node in the genomic chain).
- `Specimen.specimen_name` ← `specimen_type` from CSV
- `Specimen.class` — derived from type (plasma → biological, blood → biological)

In [ ]:
from datetime import date as _date


def _age(dob_str):
    try:
        dob = _date.fromisoformat(str(dob_str)[:10])
        today = _date.today()
        return today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))
    except Exception:
        return None


# --- Patients ---
df = pd.read_csv(CSV_DIR / "Patients.csv").fillna("")
for _, r in df.iterrows():
    run("""
        MERGE (p:Patient {patient_id: $patient_id})
        ON CREATE SET
            p.firstname     = $firstname,
            p.lastname      = $lastname,
            p.sex           = $sex,
            p.date_of_birth = date($dob),
            p.created_at    = datetime()
    """, patient_id=r.patient_id, firstname=r.firstname, lastname=r.lastname,
         sex=r.sex, dob=str(r.date_of_birth)[:10])
print(f"Patients loaded: {len(df)}")

# --- OPD Visits ---
df = pd.read_csv(CSV_DIR / "OpdVisits.csv").fillna("")
for _, r in df.iterrows():
    run("""
        MERGE (v:OpdVisit {visit_id: $visit_id})
        ON CREATE SET
            v.visit_date      = CASE WHEN $visit_date <> '' THEN date($visit_date) ELSE NULL END,
            v.clinic          = $clinic,
            v.chief_complaint = $chief_complaint
    """, visit_id=r.visit_id, visit_date=str(r.visit_date)[:10],
         clinic=str(r.clinic), chief_complaint=str(r.chief_complaint))
print(f"OPD visits loaded: {len(df)}")

# --- Admissions ---
df_adm = pd.read_csv(CSV_DIR / "Admissions.csv").fillna("")
for _, r in df_adm.iterrows():
    doa = str(r.date_of_admission)[:10] if r.date_of_admission else ""
    dod = str(r.date_of_discharge)[:10] if r.date_of_discharge else ""
    run("""
        MERGE (a:Admission {admission_id: $admission_id})
        ON CREATE SET
            a.date_of_admission = CASE WHEN $doa <> '' THEN date($doa) ELSE NULL END,
            a.date_of_discharge = CASE WHEN $dod <> '' THEN date($dod) ELSE NULL END,
            a.room_no = $room_no,
            a.bed_no  = $bed_no
    """, admission_id=r.admission_id, doa=doa, dod=dod,
         room_no=str(r.room_no), bed_no=str(r.bed_no))
print(f"Admissions loaded: {len(df_adm)}")

# --- Specimens (virowatch: specimen_name, class, category) ---
df_sp = pd.read_csv(CSV_DIR / "Specimens.csv").fillna("")
for _, r in df_sp.iterrows():
    run("""
        MERGE (s:Specimen {specimen_id: $specimen_id})
        ON CREATE SET
            s.specimen_name = $specimen_name,
            s.class         = 'biological',
            s.category      = $category
    """, specimen_id=r.specimen_id, specimen_name=r.specimen_type,
         category=r.specimen_type)
print(f"Specimens loaded: {len(df_sp)}")

---
## 4. Lab Results & HIV Viral Loads

`LabResults.csv`, `HIVViralLoads.csv`

**Virowatch property names:**
- `LabResult.result_id` ← `lab_id`; `result_date` ← `test_date`; `description` ← `result_type`
- `HIVViralLoad.result_id` ← `viral_load_id`; `value` ← `value_copies_per_ml` (INTEGER); `unit = 'copies/mL'`

In [ ]:
# --- Lab Results ---
df = pd.read_csv(CSV_DIR / "LabResults.csv").fillna("")
for _, r in df.iterrows():
    run("""
        MERGE (lr:LabResult {result_id: $result_id})
        ON CREATE SET
            lr.result_date  = CASE WHEN $result_date <> '' THEN date($result_date) ELSE NULL END,
            lr.description  = $description,
            lr.raw_value_str = $raw_value_str,
            lr.lab_name     = $lab_name,
            lr.created_at   = datetime()
    """, result_id=r.lab_id, result_date=str(r.test_date)[:10],
         description=r.result_type, raw_value_str=str(r.value), lab_name=str(r.notes))
print(f"Lab results loaded: {len(df)}")

# --- HIV Viral Loads (virowatch: result_id, value INTEGER, unit) ---
df = pd.read_csv(CSV_DIR / "HIVViralLoads.csv").fillna("")
for _, r in df.iterrows():
    run("""
        MERGE (vl:HIVViralLoad {result_id: $result_id})
        ON CREATE SET
            vl.value  = $value,
            vl.unit   = 'copies/mL',
            vl.created_at = datetime()
    """, result_id=r.viral_load_id, value=maybe_int(r.value_copies_per_ml))
print(f"HIV viral loads loaded: {len(df)}")

---
## 5. Wire Relationships

All typed, directed edges matching the virowatch schema.

In [ ]:
# Ward -[:IN_DEPARTMENT]-> Department
df = pd.read_csv(CSV_DIR / "Wards.csv")
for _, r in df.iterrows():
    run("""
        MATCH (w:Ward {ward_id: $ward_id})
        MATCH (d:Department {department_id: $department_id})
        MERGE (w)-[:IN_DEPARTMENT]->(d)
    """, ward_id=r.ward_id, department_id=r.department_id)
print(f"Ward -[:IN_DEPARTMENT]-> Department: {len(df)}")

# Patient -[:HAS_ADMISSION]-> Admission -[:ADMITTED_TO]-> Ward
df_adm = pd.read_csv(CSV_DIR / "Admissions.csv").fillna("")
for _, r in df_adm.iterrows():
    run("""
        MATCH (p:Patient {patient_id: $patient_id})
        MATCH (a:Admission {admission_id: $admission_id})
        MERGE (p)-[:HAS_ADMISSION]->(a)
    """, patient_id=r.patient_id, admission_id=r.admission_id)
    run("""
        MATCH (a:Admission {admission_id: $admission_id})
        MATCH (w:Ward {ward_id: $ward_id})
        MERGE (a)-[:ADMITTED_TO]->(w)
    """, admission_id=r.admission_id, ward_id=r.ward_id)
print(f"Admission chains wired: {len(df_adm)}")

# Patient -[:HAS_OPD_VISIT]-> OpdVisit
df_opd = pd.read_csv(CSV_DIR / "OpdVisits.csv")
for _, r in df_opd.iterrows():
    run("""
        MATCH (p:Patient {patient_id: $patient_id})
        MATCH (v:OpdVisit {visit_id: $visit_id})
        MERGE (p)-[:HAS_OPD_VISIT]->(v)
    """, patient_id=r.patient_id, visit_id=r.visit_id)
print(f"Patient -[:HAS_OPD_VISIT]-> OpdVisit: {len(df_opd)}")

# Specimen -[:COLLECTED_FROM {on}]-> Patient  (virowatch direction)
# Specimen -[:COLLECTED_AT_VISIT]-> OpdVisit
df_sp = pd.read_csv(CSV_DIR / "Specimens.csv").fillna("")
for _, r in df_sp.iterrows():
    run("""
        MATCH (s:Specimen {specimen_id: $specimen_id})
        MATCH (p:Patient {patient_id: $patient_id})
        MERGE (s)-[:COLLECTED_FROM]->(p)
    """, specimen_id=r.specimen_id, patient_id=r.patient_id)
    if r.visit_id:
        run("""
            MATCH (s:Specimen {specimen_id: $specimen_id})
            MATCH (v:OpdVisit {visit_id: $visit_id})
            MERGE (s)-[:COLLECTED_AT_VISIT]->(v)
        """, specimen_id=r.specimen_id, visit_id=r.visit_id)
print(f"Specimen -[:COLLECTED_FROM]-> Patient: {len(df_sp)}")

# Specimen -[:TESTED_FOR {on}]-> LabResult  (virowatch)
df_lr = pd.read_csv(CSV_DIR / "LabResults.csv")
for _, r in df_lr.iterrows():
    run("""
        MATCH (s:Specimen {specimen_id: $specimen_id})
        MATCH (lr:LabResult {result_id: $result_id})
        MERGE (s)-[:TESTED_FOR]->(lr)
    """, specimen_id=r.specimen_id, result_id=r.lab_id)
print(f"Specimen -[:TESTED_FOR]-> LabResult: {len(df_lr)}")

# Patient -[:HAS_HIV_VIRAL_LOAD_RESULT {date}]-> HIVViralLoad
# Specimen -[:TESTED_FOR {on}]-> HIVViralLoad  (virowatch has both)
df_vl = pd.read_csv(CSV_DIR / "HIVViralLoads.csv").fillna("")
for _, r in df_vl.iterrows():
    run("""
        MATCH (p:Patient {patient_id: $patient_id})
        MATCH (vl:HIVViralLoad {result_id: $result_id})
        MERGE (p)-[rel:HAS_HIV_VIRAL_LOAD_RESULT]->(vl)
        ON CREATE SET rel.date = CASE WHEN $test_date <> '' THEN date($test_date) ELSE NULL END
    """, patient_id=r.patient_id, result_id=r.viral_load_id,
         test_date=str(r.test_date)[:10])
print(f"Patient -[:HAS_HIV_VIRAL_LOAD_RESULT]-> HIVViralLoad: {len(df_vl)}")

# ReferenceGenome -[:OF]-> Organism  (virowatch uses OF, not REFERENCE_GENOME_OF)
df_rg = pd.read_csv(CSV_DIR / "ReferenceGenomes.csv")
for _, r in df_rg.iterrows():
    run("""
        MATCH (rg:ReferenceGenome {accession: $accession})
        MATCH (o:Organism {taxid: $taxid})
        MERGE (rg)-[:OF]->(o)
    """, accession=r.accession_no, taxid=str(r.taxid))
print(f"ReferenceGenome -[:OF]-> Organism: {len(df_rg)}")

print("\nAll relationships wired.")

---
## 6. Genomic Provenance & Variants

Virowatch anchors genomic data on the **Specimen**. The traversal spine is:

```
Specimen
  -[:SEQUENCED {on}]-> SequencingRun
  -[:YIELDED]-> Assembly
  -[:HAS_CONTIG]-> Contig
  -[:HAS_VARIANT_CALLING_RUN]-> VariantCallingRun
  -[:FOUND {DP, GT, QUAL}]-> Variant
  -[:AFFECTS]-> Feature
  -[:ON]-> Contig
```

And `Specimen -[:HAS_ASSEMBLY]-> Assembly` (direct shortcut).

**Variant property names (virowatch):** `pos` (STRING), `ref`, `alt`, `reference_accession`,
`hgvsc`, `hgvsp`, `variant_type`, `effect`, `impact` — all lowercase, matching the live DB.

In [ ]:
# ── Stub provenance nodes ──────────────────────────────────────────────────
# BAC_S001 (Samples.csv) is derived from SP003 — SP003 is the Specimen anchor.

run("""
    MERGE (sr:SequencingRun {run_id: 'DEMO_SR_001'})
    ON CREATE SET sr.platform = 'Illumina', sr.date = date('2025-02-01')
""")

run("""
    MERGE (a:Assembly {assembly_id: 'DEMO_ASM_001'})
    ON CREATE SET a.assembler = 'Flye', a.date = date('2025-02-10')
""")

run("""
    MERGE (c:Contig {contig_id: 'DEMO_CONTIG_001'})
    ON CREATE SET
        c.name        = 'contig_1',
        c.length      = 5000000,
        c.coverage    = 50,
        c.is_circular = true,
        c.is_repeated_region = false
""")

run("""
    MERGE (vcr:VariantCallingRun {run_id: 'DEMO_VCR_001'})
""")

print("Provenance stub nodes created.")

# ── Wire the provenance spine ──────────────────────────────────────────────
spine = [
    # Specimen -[:SEQUENCED {on}]-> SequencingRun
    ("""
        MATCH (sp:Specimen {specimen_id: 'SP003'})
        MATCH (sr:SequencingRun {run_id: 'DEMO_SR_001'})
        MERGE (sp)-[r:SEQUENCED]->(sr)
        ON CREATE SET r.on = date('2025-02-01')
    """, {}),
    # SequencingRun -[:YIELDED]-> Assembly
    ("""
        MATCH (sr:SequencingRun {run_id: 'DEMO_SR_001'})
        MATCH (a:Assembly {assembly_id: 'DEMO_ASM_001'})
        MERGE (sr)-[:YIELDED]->(a)
    """, {}),
    # Specimen -[:HAS_ASSEMBLY]-> Assembly (virowatch direct shortcut)
    ("""
        MATCH (sp:Specimen {specimen_id: 'SP003'})
        MATCH (a:Assembly {assembly_id: 'DEMO_ASM_001'})
        MERGE (sp)-[:HAS_ASSEMBLY]->(a)
    """, {}),
    # Assembly -[:HAS_CONTIG]-> Contig
    ("""
        MATCH (a:Assembly {assembly_id: 'DEMO_ASM_001'})
        MATCH (c:Contig {contig_id: 'DEMO_CONTIG_001'})
        MERGE (a)-[:HAS_CONTIG]->(c)
    """, {}),
    # Assembly -[:HAS_VARIANT_CALLING_RUN]-> VariantCallingRun
    ("""
        MATCH (a:Assembly {assembly_id: 'DEMO_ASM_001'})
        MATCH (vcr:VariantCallingRun {run_id: 'DEMO_VCR_001'})
        MERGE (a)-[:HAS_VARIANT_CALLING_RUN]->(vcr)
    """, {}),
    # ReferenceGenome -[:HAS_CONTIG]-> Contig
    ("""
        MATCH (rg:ReferenceGenome {accession: 'NZ_CP023401.1'})
        MATCH (c:Contig {contig_id: 'DEMO_CONTIG_001'})
        MERGE (rg)-[:HAS_CONTIG]->(c)
    """, {}),
]
for cql, params in spine:
    run(cql, **params)
print("Provenance spine wired.")

In [ ]:
# ── Features from SNPs.csv LOCUS_TAG column ───────────────────────────────
df_snp = pd.read_csv(CSV_DIR / "SNPs.csv").fillna("")

seen_loci = set()
for _, r in df_snp.iterrows():
    locus = clean(r.LOCUS_TAG)
    if not locus or locus in seen_loci:
        continue
    seen_loci.add(locus)
    run("""
        MERGE (f:Feature {locus_tag: $locus_tag})
        ON CREATE SET
            f.name                = $locus_tag,
            f.reference_accession = $reference_accession,
            f.product             = $product
        WITH f
        MATCH (rg:ReferenceGenome {accession: $reference_accession})
        MERGE (rg)-[:HAS_FEATURE]->(f)
    """, locus_tag=locus, reference_accession=r.REF_ACC,
         product=clean(r.LOCUS_TAG_ID) or "")

print(f"Features created: {len(seen_loci)}")

# ── Variants from SNPs.csv (virowatch property names) ─────────────────────
# Virowatch: pos=STRING, ref, alt, reference_accession, hgvsc, hgvsp,
#            variant_type, effect, impact
# Relationship: VariantCallingRun -[:FOUND {DP, GT, QUAL}]-> Variant

variants = [
    {
        "reference_accession": r.REF_ACC,
        "pos":          str(r.POS),          # STRING in virowatch
        "ref":          str(r.REF),
        "alt":          str(r.ALT),
        "variant_type": str(r.TYPE),
        "effect":       str(r.EFFECT),
        "impact":       str(r.IMPACT),
        "hgvsc":        clean(r.HGVS_C) or "",
        "hgvsp":        clean(r.HGVS_P) or "",
        "locus_tag":    clean(r.LOCUS_TAG) or "",
        "DP":           str(r.DP),
        "GT":           str(r.GT),
        "QUAL":         str(r.QUAL),
        "run_id":       "DEMO_VCR_001",
    }
    for _, r in df_snp.iterrows()
]

BATCH = 100
total = 0
for i in range(0, len(variants), BATCH):
    results = run("""
        UNWIND $variants AS v
        MERGE (var:Variant {
            reference_accession: v.reference_accession,
            pos: v.pos, ref: v.ref, alt: v.alt
        })
        ON CREATE SET
            var.variant_type = v.variant_type,
            var.effect       = v.effect,
            var.impact       = v.impact,
            var.hgvsc        = v.hgvsc,
            var.hgvsp        = v.hgvsp
        // VariantCallingRun -[:FOUND {DP,GT,QUAL}]-> Variant (virowatch)
        WITH var, v
        MATCH (vcr:VariantCallingRun {run_id: v.run_id})
        MERGE (vcr)-[f:FOUND]->(var)
        SET f.DP = v.DP, f.GT = v.GT, f.QUAL = v.QUAL
        // Variant -[:ON]-> Contig (demo stub)
        WITH var, v
        MATCH (c:Contig {contig_id: 'DEMO_CONTIG_001'})
        MERGE (var)-[:ON]->(c)
        // Variant -[:AFFECTS]-> Feature
        WITH var, v
        OPTIONAL MATCH (f:Feature {locus_tag: v.locus_tag})
        WITH var, f WHERE f IS NOT NULL
        MERGE (var)-[:AFFECTS]->(f)
        RETURN count(var) AS processed
    """, variants=variants[i:i + BATCH])
    total += results[0]["processed"] if results else 0

print(f"Variants merged: {total} (from {len(variants)} SNP rows)")

---
## 7. Explore the Graph

Three read queries to verify the loaded data and exercise the virowatch schema.

### 7a. Node counts by label

In [ ]:
rows = run("""
    MATCH (n)
    WITH labels(n) AS lbls, count(*) AS n
    UNWIND lbls AS lbl
    RETURN lbl AS label, sum(n) AS count
    ORDER BY count DESC
""")
pd.DataFrame([dict(r) for r in rows])

### 7b. Patient → Specimen → Variant traversal (virowatch spine)

Traverses the full virowatch chain:
`Patient ← Specimen → Assembly → VariantCallingRun → Variant`

In [ ]:
rows = run("""
    MATCH (p:Patient)<-[:COLLECTED_FROM]-(sp:Specimen)
    OPTIONAL MATCH
        (sp)-[:HAS_ASSEMBLY]->(:Assembly)
            -[:HAS_VARIANT_CALLING_RUN]->(:VariantCallingRun)
            -[:FOUND]->(v:Variant)
    RETURN
        p.patient_id                    AS patient_id,
        p.firstname + ' ' + p.lastname  AS name,
        sp.specimen_id                  AS specimen_id,
        sp.specimen_name                AS specimen_type,
        count(v)                        AS variant_count
    ORDER BY p.patient_id
""")
pd.DataFrame([dict(r) for r in rows])

### 7c. Variant detail — effect and affected feature

In [ ]:
rows = run("""
    MATCH (vcr:VariantCallingRun {run_id: 'DEMO_VCR_001'})
          -[f:FOUND]->(v:Variant)
    OPTIONAL MATCH (v)-[:AFFECTS]->(feat:Feature)
    RETURN
        v.reference_accession AS ref_acc,
        v.pos                 AS pos,
        v.ref                 AS ref,
        v.alt                 AS alt,
        v.effect              AS effect,
        v.impact              AS impact,
        v.hgvsc               AS hgvs_c,
        feat.locus_tag        AS locus_tag,
        f.DP                  AS DP,
        f.QUAL                AS QUAL
    ORDER BY toInteger(v.pos)
    LIMIT 20
""")
pd.DataFrame([dict(r) for r in rows])

### 7d. HIV patient viral load timeline

In [ ]:
rows = run("""
    MATCH (p:Patient)-[rel:HAS_HIV_VIRAL_LOAD_RESULT]->(vl:HIVViralLoad)
    RETURN
        p.patient_id                   AS patient_id,
        p.firstname + ' ' + p.lastname AS name,
        toString(rel.date)             AS test_date,
        vl.value                       AS copies_per_ml,
        vl.unit                        AS unit
    ORDER BY p.patient_id, rel.date
""")
pd.DataFrame([dict(r) for r in rows])

---

**Done.** The graph now holds all clinical and genomic data from `example/csv/` in the virowatch-compatible schema.

Schema summary:
- **Specimen** is the genomic anchor (not Sample)
- Genomic spine: `Specimen →[:SEQUENCED]→ SequencingRun →[:YIELDED]→ Assembly →[:HAS_VARIANT_CALLING_RUN]→ VariantCallingRun →[:FOUND]→ Variant →[:AFFECTS]→ Feature`
- `Specimen →[:HAS_ASSEMBLY]→ Assembly` shortcut available
- `ReferenceGenome →[:OF]→ Organism` (not `:REFERENCE_GENOME_OF`)

Close the driver when finished:

In [ ]:
driver.close()
print("Driver closed.")